# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-5.1'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [01:54<00:00, 22.81s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

"Title: Costway 330-lbs. Folding Platform Hand Truck for $35 + free shipping w/ $35\nDetails: It's the best price we've seen for it and $25 less than what you'd pay elsewhere. Opt for pickup to avoid the $6.99 shipping charge (or get free shipping with orders of $35 or more). Buy Now at Walmart\nFeatures: \nURL: https://www.dealnews.com/products/Costway/Costway-330-lbs-Folding-Platform-Hand-Truck/488025.html?iref=rss-c196"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Best Buy Holiday Apple AirPod Deals: Up to $120 off + free shipping
Details: Save on a collection of in-ear and over-ear headphones from the Apple. We've pictured the Apple AirPods Max Wireless Over-Ear Headphones (USB-C) for $429.99 ($549 elsewhere). My Best Buy members get free shipping. (It's free to join. Shipping is free for everyone over $35. Pickup may also b

In [13]:

def get_recommendations():
    resp = openai.responses.parse(
        model="gpt-5.1",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=DealSelection,
    )
    return resp.output_parsed # or keep the whole list if you expect multiple


In [14]:
result = get_recommendations()

In [15]:
len(result.deals)

5

In [16]:
result.deals[1]

Deal(product_description='The unlocked Samsung Galaxy S24 Ultra 512GB Android smartphone is a high-end flagship device built around the Qualcomm Snapdragon 8 Gen 3 processor for fast performance and efficient multitasking. It features a large 6.8-inch Dynamic AMOLED 2X HD+ display, delivering deep blacks, high brightness, and smooth scrolling that’s ideal for gaming, streaming, and productivity. With 12GB of RAM and 512GB of internal storage, it can handle demanding apps and store extensive photo, video, and app libraries without compromise. The camera system supports up to 200MP zoom capabilities, targeting users who want advanced photography and detail-rich images, while Galaxy AI features assist with tasks like image editing and productivity. Being unlocked, the phone can be used with a wide range of carriers, and this model (SM-S928UZKFXAA) is suited to power users who need cutting-edge hardware in a premium build.', price=563.0, url='https://www.dealnews.com/products/Samsung/Unloc

In [17]:
from agents.scanner_agent import ScannerAgent

In [18]:
agent = ScannerAgent()
result = agent.scan()

In [20]:
print(result)

deals=[Deal(product_description='The Apple AirPods Max are premium wireless over-ear headphones that provide an immersive audio experience with high-fidelity sound. Designed for comfort, they feature a breathable knit mesh canopy and memory foam ear cushions to ensure a perfect fit for prolonged use. The seamless integration with Apple devices allows for automatic switching and spatial audio, making them ideal for music lovers and movie watchers alike. With a price point of $429.99, these headphones represent a significant investment in listening quality.', price=429.99, url='https://www.dealnews.com/Best-Buy-Holiday-Apple-Air-Pod-Deals-Up-to-120-off-free-shipping/21793589.html?iref=rss-c142'), Deal(product_description="The Samsung Galaxy Watch FE is a 40mm smartwatch that combines style with functionality. It features an AMOLED display making it vibrant and easy to read, along with health monitoring capabilities such as ECG and real-time heart rate readings. With a battery life of 40 